In [1]:
import torch
torch.set_default_dtype(torch.float64)
from cpu_cb import E8P12_codebook

In [2]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    b = torch.tensor([1,1,-1,-1]).to(x.dtype)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1) + (4,) + (1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k) / (2**k)

In [3]:
def calculate_k(n):
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    return k

In [4]:
def hb_transform_loop(x, k): # hb_transform without the end permutate and reshape
    (m,n) = x.shape
    assert 4**k == n
    b = torch.tensor([1, 1, -1, -1]).to(x.dtype)
    x = x.reshape((m,) + (4,) * k)
    for i in range(k):
        x = x.flip(1 + i) + x * b.view((1,) * (i + 1) + (4,) + (1,) * (k - 1 - i))
    # return x.reshape((m,) + (2, 2) * k) / (2**k)
    return x.reshape((m,) + (2, 2) * k)

In [5]:
def hb_transform_reshape(x, k): # given original hb_transform function shape go to the hb_transform_loop
    m = x.shape[0]
    x = x.reshape((m,) + (2,) * (2 * k))
    forward_perm = [0] + [2 * i + 1 for i in range(k)] + [2 * i + 2 for i in range(k)]
    inverse_perm = [0] * (2 * k + 1)
    for i, p in enumerate(forward_perm):
        inverse_perm[p] = i 
    x = x.permute(inverse_perm)
    return x.reshape((m,) + (4,) * k)

In [6]:
cliques = [
    [0, 2, 11, 25, 33, 39, 47, 57],
    [1, 3, 10, 24, 32, 38, 46, 56],
    [4, 6, 15, 29, 35, 37, 43, 61],
    [5, 7, 14, 28, 34, 36, 42, 60],
    [12, 18, 20, 26, 44, 53, 55, 62],
    [13, 19, 21, 27, 45, 52, 54, 63],
    [8, 16, 22, 30, 40, 49, 51, 58],
    [9, 17, 23, 31, 41, 48, 50, 59]
]
cb = E8P12_codebook()

In [7]:
def bt(M, n):
    k = calculate_k(n * n)
    M_flat = hb_transform_reshape(M.reshape(1, n, n), k).reshape(1, n * n)
    result = hb_transform_loop(M_flat, k) / (2**k)
    return result.reshape(n * n) / (1.0 / n)

def ibt(error, n):
    return hb_transform(error.reshape(1, n * n)).reshape(n, n)

In [8]:
def bulk_orth_quant(coeffs, H_sqrt, n, iters):
    tr_H_sqrt = H_sqrt.diagonal().sum()
    base_cliques = torch.tensor(cliques, dtype=torch.long)
    num_cliques = n*n // 64
    offsets = torch.arange(num_cliques, dtype=torch.long).view(-1, 1, 1) * 64
    all_cliques = offsets + base_cliques.unsqueeze(0)
    all_cliques = all_cliques.reshape(-1, 8)
    hat_coeffs = torch.zeros_like(coeffs)
    for c in all_cliques:
        hat_coeffs[c] = cb.quantize(coeffs[c].unsqueeze(0))[0].squeeze(0)
    for _ in range(iters):
        error = coeffs - hat_coeffs
        correction = bt(ibt(error, n) @ H_sqrt, n) / tr_H_sqrt
        corrected = coeffs + correction
        hat_new = torch.zeros_like(coeffs)
        for c in all_cliques:
            hat_new[c] = cb.quantize(corrected[c].unsqueeze(0))[0].squeeze(0)
        if torch.allclose(hat_new, hat_coeffs):
            break
        hat_coeffs = hat_new
    return hat_coeffs

In [9]:
n = 64
k = calculate_k(n * n)
# U_W, _ = torch.linalg.qr(torch.randn(n, n))
# V_W, _ = torch.linalg.qr(torch.randn(n, n))
# lambda_W = torch.tensor([1.0 / i**2 for i in range(1, n+1)], dtype=torch.float64)
# W = U_W @ torch.diag(lambda_W) @ V_W.T
W = torch.randn(n, n)
# U_H, _ = torch.linalg.qr(torch.randn(n, n))
# lambda_H = torch.tensor([1.0 / i for i in range(1, n+1)], dtype=torch.float64)
# H = U_H @ torch.diag(lambda_H) @ U_H.T
H = torch.randn(n,n)
H = H @ H.t()
H = (H + H.t())/2
H.diagonal().add_(H.diagonal().mean() * 0.01)

tensor([58.2855, 72.4726, 56.0864, 61.8714, 67.2898, 50.5102, 84.3998, 66.2338,
        65.8473, 59.5346, 52.1131, 74.0634, 42.4869, 63.4968, 74.6973, 74.4161,
        61.1000, 67.8425, 67.4299, 81.2833, 53.9692, 73.7432, 56.6082, 43.7353,
        51.0820, 88.9913, 54.5181, 74.2577, 67.7883, 62.1493, 65.1518, 46.9967,
        70.4772, 57.7950, 82.2565, 72.9593, 58.1966, 58.6999, 63.0534, 82.2030,
        62.2317, 68.2338, 48.6184, 73.5638, 40.6331, 80.4339, 52.9178, 91.1996,
        72.6041, 63.0768, 59.8166, 69.1688, 75.8816, 76.4852, 36.1986, 56.5212,
        84.0231, 72.3078, 78.8189, 55.0587, 47.4724, 73.0723, 61.4522, 83.4085])

In [10]:
print("Adaptive rounding")
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = bulk_orth_quant(coeffs, H_sqrt, n, 100)
hat_coeffs = hat_coeffs * Wscale
hatW_bulk = hb_transform(hat_coeffs.reshape(1, n*n)).reshape(n, n)
err = (W - hatW_bulk).norm() / W.norm()
print(f"Error: {err}")
proxy_loss = ((hatW_bulk - W) @ H @ (hatW_bulk - W).T).trace()
print(f"Proxy loss: {proxy_loss}")

Adaptive rounding
Error: 0.30373881333174063
Proxy loss: 26764.85865690799


In [11]:
print("Adaptive rounding")
H_nr = torch.eye(n)
S, V = torch.linalg.eigh(H_nr)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
W_flat = W.reshape(1, 2**k, 2**k)
W_reshaped = hb_transform_reshape(W_flat, k).reshape(1, n*n)
coeffs = hb_transform_loop(W_reshaped, k) / (2**k)
coeffs = coeffs.reshape(n*n)
norm = 1/n
coeffs = coeffs / norm
Wscale = coeffs.square().mean().sqrt() / cb.opt_scale
coeffs = coeffs / Wscale
hat_coeffs = bulk_orth_quant(coeffs, H_sqrt, n, 100)
hat_coeffs = hat_coeffs * Wscale
hatW_bulk = hb_transform(hat_coeffs.reshape(1, n*n)).reshape(n, n)
err = (W - hatW_bulk).norm() / W.norm()
print(f"Error: {err}")
proxy_loss = ((hatW_bulk - W) @ H @ (hatW_bulk - W).T).trace()
print(f"Proxy loss: {proxy_loss}")

Adaptive rounding
Error: 0.3037103622216449
Proxy loss: 26841.737190958225
